In [9]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error, r2_score, mean_squared_error

PROJECT_ROOT = Path("..")   # 你的 notebook 在 notebooks/ 里
DATA_DIR = PROJECT_ROOT / "data"
ART_DIR = PROJECT_ROOT / "results" / "model_artifacts"

DATA_DIR, ART_DIR


(PosixPath('../data'), PosixPath('../results/model_artifacts'))

In [10]:
from pathlib import Path

def find_project_root(start: Path):
    cur = start.resolve()
    for _ in range(6):
        if (cur / "data" / "df_feat.parquet").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root containing data/df_feat.parquet")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
ART_DIR = PROJECT_ROOT / "results" / "model_artifacts"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_DIR exists:", ART_DIR.exists())
sorted([p.name for p in ART_DIR.glob("*")])[:10]


PROJECT_ROOT: /Users/wenxi/Desktop/TFM_25
ART_DIR exists: True


['clf_ed_xgb.joblib',
 'clf_ed_xgb.meta.json',
 'clf_highcost_rf.joblib',
 'clf_highcost_rf.meta.json',
 'clf_ip_rf.joblib',
 'clf_ip_rf.meta.json',
 'reg_log_totexpy2_xgb_es.booster.json',
 'reg_log_totexpy2_xgb_es.meta.json',
 'reg_log_totexpy2_xgb_es.preprocess.joblib']

In [11]:
df_feat = pd.read_parquet(DATA_DIR / "df_feat.parquet")
df_feat.shape, df_feat.columns[:10]


((7812, 141),
 Index(['DUID', 'PID', 'DUPERSID', 'PANEL', 'YEARIND', 'ALL5RDS', 'DIED',
        'INST', 'MILITARY', 'ENTRSRVY'],
       dtype='object'))

In [12]:
for c in ["LONGWT", "VARSTR", "VARPSU"]:
    print(c, c in df_feat.columns, df_feat[c].isna().sum() if c in df_feat.columns else None)


LONGWT True 0
VARSTR True 0
VARPSU True 0


In [13]:
import sys
sys.path.append(str(PROJECT_ROOT))
from src.models import split_train_val_test



#  saved 3 classification pipeline + meta（with threshold）

In [14]:
# HIGHCOST RF
clf_highcost = joblib.load(ART_DIR / "clf_highcost_rf.joblib")
meta_highcost = json.loads((ART_DIR / "clf_highcost_rf.meta.json").read_text())
t_highcost = meta_highcost["best_threshold"]

# ED XGB
clf_ed = joblib.load(ART_DIR / "clf_ed_xgb.joblib")
meta_ed = json.loads((ART_DIR / "clf_ed_xgb.meta.json").read_text())
t_ed = meta_ed["best_threshold"]

# IP RF
clf_ip = joblib.load(ART_DIR / "clf_ip_rf.joblib")
meta_ip = json.loads((ART_DIR / "clf_ip_rf.meta.json").read_text())
t_ip = meta_ip["best_threshold"]

t_highcost, t_ed, t_ip


/opt/anaconda3/envs/meps/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/meps/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/meps/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.8.0. Th

(0.5499999999999999, 0.2, 0.44999999999999996)

# saved regression model: booster + preprocess + best_iter

In [15]:
pre_reg = joblib.load(ART_DIR / "reg_log_totexpy2_xgb_es.preprocess.joblib")

booster = xgb.Booster()
booster.load_model(ART_DIR / "reg_log_totexpy2_xgb_es.booster.json")

meta_reg = json.loads((ART_DIR / "reg_log_totexpy2_xgb_es.meta.json").read_text())
best_iter = int(meta_reg["best_iteration"])

best_iter


1037

# classfication: weighted AUC/PR-AUC + weighted precision/recall

In [16]:
def weighted_clf_metrics(y_true, proba, w, threshold):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    w = np.asarray(w).astype(float)

    auc_w = roc_auc_score(y_true, proba, sample_weight=w)
    pr_w = average_precision_score(y_true, proba, sample_weight=w)

    pred = (proba >= threshold).astype(int)
    tp = w[(y_true==1) & (pred==1)].sum()
    fp = w[(y_true==0) & (pred==1)].sum()
    fn = w[(y_true==1) & (pred==0)].sum()

    precision_w = tp / (tp + fp + 1e-12)
    recall_w = tp / (tp + fn + 1e-12)

    return {"AUC_w": float(auc_w), "PR_AUC_w": float(pr_w),
            "precision_w": float(precision_w), "recall_w": float(recall_w)}


# regression：weighted RMSE/MAE/R²

In [17]:
def weighted_reg_metrics(y_true, y_pred, w):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    w = np.asarray(w).astype(float)

    rmse_w = np.sqrt(mean_squared_error(y_true, y_pred, sample_weight=w))
    mae_w = mean_absolute_error(y_true, y_pred, sample_weight=w)
    r2_w = r2_score(y_true, y_pred, sample_weight=w)
    return {"RMSE_w": float(rmse_w), "MAE_w": float(mae_w), "R2_w": float(r2_w)}


# define “split + weight” helper

In [18]:
def split_with_weights(df, target, feature_cols, *, random_state=42, stratify=False,
                       w_col="LONGWT", str_col="VARSTR", psu_col="VARPSU"):
    tmp = df[feature_cols + [target, w_col, str_col, psu_col]].dropna()

    X = tmp[feature_cols].copy()
    y = tmp[target].copy()

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )

    w_test = tmp.loc[X_test.index, w_col].astype(float).values
    return X_test, y_test, w_test


# weighted evaluation（on test ）

**HIGHCOST_Y2**

In [ ]:
cols_hc = list(clf_highcost.feature_names_in_)
X_test, y_test, w_test = split_with_weights(df_feat, "HIGHCOST_Y2", cols_hc, random_state=42, stratify=True)

proba = clf_highcost.predict_proba(X_test[cols_hc])[:, 1]
hc_w = weighted_clf_metrics(y_test.values, proba, w_test, threshold=t_highcost)
hc_w
